# Task 2

In [12]:
# # clear unneeded folders
# classes = None
# with open("data/food-101/meta/classes.txt") as f:
#     classes = f.readlines()
# classes = [c.strip("\n") for c in classes]
# print(len(classes))
# classes.remove("hot_and_sour_soup")
# classes.remove("pork_chop")
# classes.remove("beef_tartare")
# classes.remove("crab_cakes")
# classes.remove("baby_back_ribs")
# classes.remove("guacamole")
# classes.remove("french_onion_soup")
# classes.remove("dumplings")
# classes.remove("sushi")
# classes.remove("cheesecake")
# print(len(classes))

# import os
# import shutil

# current_dir = os.getcwd()
# images_dir = os.path.join(current_dir, "data/Food10")
# for c in classes:
#     to_rm = os.path.join(images_dir, c)
#     shutil.rmtree(to_rm)


In [13]:
# import os
# import shutil
# current_dir = os.getcwd()
# train_dir = os.path.join(current_dir, "Food10")
# test_dir = os.path.join(current_dir, "FoodTest")

# # Separate train and test data
# with open("test.csv") as f:
#     test_images = f.readlines()[1:]
# test_images = [ti.strip("\n").split(",")[1].split("\\") for ti in test_images]
# test_images = ["/".join(ti[-2:]) for ti in test_images]

# for im in test_images:
#     shutil.move(os.path.join(train_dir, im), os.path.join(test_dir, im))

# # Food10 for training, FoodTest for testing

In [14]:
import matplotlib.pyplot as plt 
import numpy as np 
import tensorflow as tf 
from tensorflow import keras 
from tensorflow.keras import layers 
from tensorflow.keras.models import Sequential 


In [15]:
import os
current_dir = os.getcwd()
train_dir = os.path.join(current_dir, "Food10")
test_dir = os.path.join(current_dir, "FoodTest")

train = tf.keras.utils.image_dataset_from_directory( 
	train_dir, 
    validation_split=0.3, 
	subset="training", 
	seed=55,
	image_size=(224, 224), 
	batch_size=32,
    label_mode='categorical') 

test = tf.keras.utils.image_dataset_from_directory( 
	train_dir, 
    validation_split=0.2, 
	subset="validation", 
	seed=55,
	image_size=(224, 224), 
	batch_size=32,
    label_mode='categorical') 

class_names = train.class_names 
print(class_names)

Found 7500 files belonging to 10 classes.
Using 5250 files for training.
Found 7500 files belonging to 10 classes.
Using 1500 files for validation.
['baby_back_ribs', 'beef_tartare', 'cheesecake', 'crab_cakes', 'dumplings', 'french_onion_soup', 'guacamole', 'hot_and_sour_soup', 'pork_chop', 'sushi']


In [16]:
# losses = [tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
#           tf.keras.losses.CategoricalCrossentropy(from_logits=True),
#           tf.keras.losses.CategoricalFocalCrossentropy(from_logits=True)]
# epoch_nums = [10, 50, 100]

# best_model = None
# best_test_accuracy = 0.0
# best_loss = None
# best_epochs = None

# try:
#     for loss in losses:
#         for epochs in epoch_nums:
#             model = Sequential([ 
#                 layers.Rescaling(1./255, input_shape=(224,224, 3)), 
#                 layers.Conv2D(16, 3, padding='same', activation='relu'), 
#                 layers.MaxPooling2D(), 
#                 layers.Conv2D(32, 3, padding='same', activation='relu'), 
#                 layers.MaxPooling2D(), 
#                 layers.Conv2D(64, 3, padding='same', activation='relu'), 
#                 layers.MaxPooling2D(), 
#                 layers.Flatten(), 
#                 layers.Dense(128, activation='relu'), 
#                 layers.Dense(10) 
#             ]) 
#             model.compile(optimizer='adam', loss=loss, metrics=['accuracy'])
#             history = model.fit(train, validation_data=test, epochs=epochs)
#             test_acc = history.history['val_accuracy'] 
#             print(test_acc)
#             if test_acc[-1] > best_test_accuracy:
#                 best_model = model
#                 best_test_accuracy = test_acc[-1]
#                 best_loss = loss
#                 best_epochs = epochs
# except KeyboardInterrupt:
#     pass
# finally:
#     print(f"Best validation accuracy achieved: {best_test_accuracy}")
#     # Dump best model just in case of accidental loss
#     import pickle
#     try:
#         with open ("best_model", "w") as f:
#             pickle.dump((best_model, best_loss, best_epochs), f)
#     except:
#         print("ERROR IN DUMPING")
#         with open("best_model", "w") as f:
#             pickle.dump(best_model, f)

In [17]:
def make_model():
    inputs = keras.Input(shape=(224,224, 3))

    # Entry block
    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.RandomFlip("horizontal")(x) # Helps introduce variety to expand dataset & reduce overfitting
    x = layers.RandomRotation(0.1)(x)
    x = layers.Conv2D(128, 3, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    previous_block_activation = x  # Set aside residual

    for size in [256, 512, 728]:
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

        # Project residual
        residual = layers.Conv2D(size, 1, strides=2, padding="same")(
            previous_block_activation
        )
        x = layers.add([x, residual])  # Add back residual
        previous_block_activation = x  # Set aside next residual

    x = layers.SeparableConv2D(1024, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.25)(x)
    # We specify activation=None so as to return logits
    outputs = layers.Dense(10, activation=None)(x)
    return keras.Model(inputs, outputs)



In [18]:
# model = make_model()
# model.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
# history = model.fit(train, validation_data=test, epochs=60)


In [19]:

# acc = history.history['accuracy'] 
# val_acc = history.history['val_accuracy'] 
# loss = history.history['loss'] 
# val_loss = history.history['val_loss'] 
# epochs_range = range(60) 
# plt.figure(figsize=(8, 8)) 
# plt.subplot(1, 2, 1) 
# plt.plot(epochs_range, acc, label='Training Accuracy') 
# plt.plot(epochs_range, val_acc, label='Validation Accuracy') 
# plt.legend(loc='lower right') 
# plt.title('Training and Validation Accuracy for Sparse Categorical Crossentropy') 
# plt.subplot(1, 2, 2) 
# plt.plot(epochs_range, loss, label='Training Loss') 
# plt.plot(epochs_range, val_loss, label='Validation Loss') 
# plt.legend(loc='upper right') 
# plt.title('Training and Validation Loss') 
# plt.show() 

In [20]:
# model = make_model()
# model.compile(optimizer='adam', loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
# history = model.fit(train, validation_data=test, epochs=60)

In [21]:
# acc = history.history['accuracy'] 
# val_acc = history.history['val_accuracy'] 
# loss = history.history['loss'] 
# val_loss = history.history['val_loss'] 
# epochs_range = range(60) 
# plt.figure(figsize=(8, 8)) 
# plt.subplot(1, 2, 1) 
# plt.plot(epochs_range, acc, label='Training Accuracy') 
# plt.plot(epochs_range, val_acc, label='Validation Accuracy') 
# plt.legend(loc='lower right') 
# plt.title('Training and Validation Accuracy for Categorical Crossentropy') 


In [22]:
# train = tf.keras.utils.image_dataset_from_directory( 
# 	train_dir, 
#     validation_split=0.3, 
# 	subset="training", 
# 	seed=55,
# 	image_size=(224, 224), 
# 	batch_size=32,
#     label_mode='categorical') 

# test = tf.keras.utils.image_dataset_from_directory( 
# 	train_dir, 
#     validation_split=0.2, 
# 	subset="validation", 
# 	seed=55,
# 	image_size=(224, 224), 
# 	batch_size=32,
#     label_mode='categorical') 

In [23]:
# model = make_model()
# model.compile(optimizer='adam', loss=tf.keras.losses.CategoricalFocalCrossentropy(from_logits=True), metrics=['accuracy'])
# history = model.fit(train, validation_data=test, epochs=50)

# acc = history.history['accuracy'] 
# val_acc = history.history['val_accuracy'] 
# loss = history.history['loss'] 
# val_loss = history.history['val_loss'] 
# epochs_range = range(100) 
# plt.figure(figsize=(8, 8)) 
# plt.subplot(1, 2, 1) 
# plt.plot(epochs_range, acc, label='Training Accuracy') 
# plt.plot(epochs_range, val_acc, label='Validation Accuracy') 
# plt.legend(loc='lower right') 
# plt.title('Training and Validation Accuracy for Categorical Focal Crossentropy') 
# plt.subplot(1, 2, 2) 
# plt.plot(epochs_range, loss, label='Training Loss') 
# plt.plot(epochs_range, val_loss, label='Validation Loss') 
# plt.legend(loc='upper right') 
# plt.title('Training and Validation Loss') 
# plt.show() 

In [ ]:
all_train = tf.keras.utils.image_dataset_from_directory( 
	train_dir, 
	image_size=(224, 224), 
	batch_size=32,
    label_mode='categorical') 

test = tf.keras.utils.image_dataset_from_directory( 
	test_dir, 
	image_size=(224, 224), 
	batch_size=32,
    label_mode='categorical') 

Found 7500 files belonging to 10 classes.
Found 2500 files belonging to 10 classes.
['baby_back_ribs', 'beef_tartare', 'cheesecake', 'crab_cakes', 'dumplings', 'french_onion_soup', 'guacamole', 'hot_and_sour_soup', 'pork_chop', 'sushi']


In [25]:
model = make_model()
model.compile(optimizer='adam', loss=tf.keras.losses.CategoricalFocalCrossentropy(from_logits=True), metrics=['accuracy'])
history = model.fit(train, epochs=40)

Epoch 1/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 182s 1s/step - accuracy: 0.2859 - loss: 0.3966
Epoch 2/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 182s 1s/step - accuracy: 0.3987 - loss: 0.3126
Epoch 3/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 182s 1s/step - accuracy: 0.4455 - loss: 0.2788
Epoch 4/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 182s 1s/step - accuracy: 0.4890 - loss: 0.2511
Epoch 5/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 182s 1s/step - accuracy: 0.5324 - loss: 0.2287
Epoch 6/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 182s 1s/step - accuracy: 0.5467 - loss: 0.2163
Epoch 7/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 181s 1s/step - accuracy: 0.5970 - loss: 0.1909
Epoch 8/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 181s 1s/step - accuracy: 0.6303 - loss: 0.1728
Epoch 9/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 181s 1s/step - accuracy: 0.6410 - loss: 0.1667
Epoch 10/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 182s 1s/step - accuracy: 0.6627 - loss: 0.1551
Epoch 11/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 183s 1s/step - accuracy: 0.6760 - loss: 0.1505
Epoch 12/40
165/165 ━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
lines = None
with open("test.csv", "r") as f:
    lines = f.readlines()[1:]
lines = [l.strip("\n").split(",") for l in lines]
txt = "original_index,label_name\n"
class_names = train.class_names 
for pair in lines:
    txt += f"{pair[0]},"
    parts = pair[1].split("\\")
    path = os.path.join(os.path.join(test_dir, parts[-2]), parts[-1])

    image = keras.utils.load_img(path, target_size=(224, 224))
    input_arr = keras.utils.img_to_array(image)
    input_arr = np.array([input_arr])  # Convert single image to a batch.
    predictions = model.predict(input_arr)[0]
    ind = np.where(predictions == max(predictions))[0][0]
    label = class_names[ind]
    txt += f"{label}\n"

with open("results.csv", "w") as f:
    f.write(txt)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━

In [29]:
result = model.predict(test)
import pickle
with open("RESULTS", "wb") as f:
    pickle.dump(result, f)

79/79 ━━━━━━━━━━━━━━━━━━━━ 17s 214ms/step
